# Dengue vs Malaria — Gold-Standard Pipeline (Decision Tree template, Baseline + SMOTE + Borderline-SMOTE + ADASYN)

Built from **raw** `cleaned_dataset_28_5_25.csv` (`MONTH, AGE, SEX, WARD, BOROUGH,
PLACE OF TREATMENT, CLASS`). Every encoding step happens inside the pipeline, so nothing
is trusted from an earlier, unaudited script.

**Encoding choices and why each is (or isn't) a leakage risk:**

| Field | Encoding | Uses `y`? | Leakage risk | Where fit |
|---|---|---|---|---|
| `MONTH` | cyclic sin/cos (fixed calendar math) | No | None | Stateless, safe globally |
| `SEX` | binary map F/M -> 0/1 | No | None | Stateless, safe globally |
| `PLACE OF TREATMENT` | binary map HOME/HOSPITAL -> 0/1 | No | None | Stateless, safe globally |
| `BOROUGH` | one-hot (16 categories) | No | None | Fit on train fold (best practice, not strictly required) |
| `WARD` | target encoding (smoothed) | **Yes** | **High if done wrong** | Fit ONLY on train fold, refit every CV fold |
| `AGE` | Z-score scaling | No | None | Fit on train fold (best practice) |

Everything except `WARD` target-encoding is deterministic and doesn't depend on the label,
so it can't leak information even if computed globally -- but we still fit it per-fold
inside the pipeline as good practice and for a clean, defensible Methods description.

This notebook covers FOUR arms: **no-resampling baseline**, **SMOTE**,
**Borderline-SMOTE**, and **ADASYN**, using the same encoding backbone, so results are
directly comparable.

**Per-model use:** only the classifier + param grid cells need to change for
RF / XGBoost / KNN / LR / MLP / GaussianNB.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score,
    roc_auc_score, average_precision_score
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN


## 1. Load RAW data

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Dengee_Maleria_Modified_XAI/cleaned_dataset_28_5_25.csv')

print('Shape:', df.shape)
print('Class distribution:', df['CLASS'].value_counts().to_dict())
df.head()


In [ ]:
X = df.drop('CLASS', axis=1)
y = df['CLASS']

print(X.dtypes)


## 2. Custom stateless encoders (deterministic -- no `y` involved, safe anywhere)

In [ ]:
class MonthCyclicEncoder(BaseEstimator, TransformerMixin):
    """Maps calendar month name -> (sin, cos) on the unit circle. Purely deterministic;
    does not use y, so there is no leakage risk regardless of when/where it's fit."""
    MONTH_MAP = {
        'JANUARY': 1, 'FEBRUARY': 2, 'MARCH': 3, 'APRIL': 4, 'MAY': 5, 'JUNE': 6,
        'JULY': 7, 'AUGUST': 8, 'SEPTEMBER': 9, 'OCTOBER': 10, 'NOVEMBER': 11, 'DECEMBER': 12,
    }

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        months = pd.Series(np.asarray(X).ravel())
        m_num = months.map(self.MONTH_MAP).astype(float)
        sin = np.sin(2 * np.pi * m_num / 12)
        cos = np.cos(2 * np.pi * m_num / 12)
        return np.column_stack([sin, cos])

    def get_feature_names_out(self, input_features=None):
        return np.array(['MONTH_sin', 'MONTH_cos'])


class BinaryMapEncoder(BaseEstimator, TransformerMixin):
    """Deterministic binary mapping for a fixed 2-category field (e.g. SEX, PLACE OF
    TREATMENT). Does not use y -- no leakage risk."""
    def __init__(self, mapping, output_name='FEATURE'):
        self.mapping = mapping
        self.output_name = output_name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        vals = pd.Series(np.asarray(X).ravel())
        return vals.map(self.mapping).astype(float).to_numpy().reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.array([self.output_name])


class WardTargetEncoder(BaseEstimator, TransformerMixin):
    """Leakage-safe target encoder. Fits ONLY on data passed to fit(X, y). Unseen wards
    at transform time fall back to the global training mean. Smoothing shrinks low-count
    wards toward the global mean."""
    def __init__(self, smoothing=10.0):
        self.smoothing = smoothing

    def fit(self, X, y):
        ward = pd.Series(np.asarray(X).ravel(), name='ward')
        target = pd.Series(np.asarray(y).ravel(), name='target')
        frame = pd.concat([ward, target], axis=1)
        self.global_mean_ = float(target.mean())
        agg = frame.groupby('ward')['target'].agg(['mean', 'count'])
        smoothed = (agg['mean'] * agg['count'] + self.global_mean_ * self.smoothing) \
                   / (agg['count'] + self.smoothing)
        self.mapping_ = smoothed.to_dict()
        return self

    def transform(self, X):
        ward = pd.Series(np.asarray(X).ravel(), name='ward')
        return ward.map(self.mapping_).fillna(self.global_mean_).to_numpy().reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.array(['WARD_TE'])


## 3. Full preprocessing pipeline

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('ward_te', SkPipeline([
        ('encode', WardTargetEncoder(smoothing=10.0)),
        ('scale', StandardScaler()),
    ]), ['WARD']),

    ('month', MonthCyclicEncoder(), ['MONTH']),

    ('age', StandardScaler(), ['AGE']),

    ('sex', BinaryMapEncoder({'F': 0, 'M': 1}, output_name='SEX'), ['SEX']),

    ('place', BinaryMapEncoder(
        {'HOME TREATMENT': 0, 'HOSPITAL TREATMENT': 1}, output_name='PLACE_OF_TREATMENT'
    ), ['PLACE OF TREATMENT']),

    ('borough', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['BOROUGH']),
])


## 4. Outer train/test split -- test set stays real and untouched

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Train class distribution:', y_train.value_counts().to_dict())
print('Test class distribution (real, never resampled):', y_test.value_counts().to_dict())


## 5a. BASELINE arm (no resampling) -- MODEL DEFINITION, swap per model

In [ ]:
clf = DecisionTreeClassifier(random_state=42)

param_grid = {
    'clf__max_depth': [None, 5, 10, 15, 20],
    'clf__min_samples_split': [2, 5, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__criterion': ['gini', 'entropy'],
}

baseline_pipeline = SkPipeline([
    ('prep', preprocessor),
    ('clf', clf),
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_baseline = GridSearchCV(
    estimator=baseline_pipeline,
    param_grid=param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
)
grid_baseline.fit(X_train, y_train)

print('Best params:', grid_baseline.best_params_)
print('Best CV F1 :', grid_baseline.best_score_)

best_baseline = grid_baseline.best_estimator_


## 5b. Baseline -- evaluate on FULL test set

In [ ]:
def full_evaluation(model, X_test, y_test, title):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f'--- {title} ---')
    print('Accuracy :', round(accuracy_score(y_test, y_pred) * 100, 2))
    print('ROC-AUC  :', round(roc_auc_score(y_test, y_proba), 4))
    print('PR-AUC   :', round(average_precision_score(y_test, y_proba), 4))
    print()
    print(classification_report(y_test, y_pred, target_names=['Dengue (0)', 'Malaria (1)']))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Dengue (0)', 'Malaria (1)'],
                yticklabels=['Dengue (0)', 'Malaria (1)'])
    plt.title(title)
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()
    return y_pred, y_proba

_ = full_evaluation(best_baseline, X_test, y_test, 'Baseline (no resampling) -- full test set')


## 6. SMOTE arm -- fit only on training fold, refit inside every CV fold

In [ ]:
smote_pipeline = ImbPipeline([
    ('prep', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('clf', DecisionTreeClassifier(random_state=42)),
])

grid_smote = GridSearchCV(
    estimator=smote_pipeline,
    param_grid=param_grid,   # same grid, 'clf__' prefix already matches
    cv=skf,
    scoring='f1',
    n_jobs=-1,
)
grid_smote.fit(X_train, y_train)

print('Best params:', grid_smote.best_params_)
print('Best CV F1 :', grid_smote.best_score_)

best_smote = grid_smote.best_estimator_

_ = full_evaluation(best_smote, X_test, y_test, 'SMOTE (training fold only) -- full real test set')


## 6b. Borderline-SMOTE arm -- same rule: fit only on training fold

Identical setup to plain SMOTE, but using `BorderlineSMOTE`, which only synthesizes new points near the decision boundary (minority samples close to majority neighbors) rather than uniformly across the whole minority class. Same leakage-safety guarantee: the test set is never touched, and every inner CV fold gets its own fresh resampling.

In [ ]:
bsmote_pipeline = ImbPipeline([
    ('prep', preprocessor),
    ('bsmote', BorderlineSMOTE(random_state=42)),
    ('clf', DecisionTreeClassifier(random_state=42)),
])

grid_bsmote = GridSearchCV(
    estimator=bsmote_pipeline,
    param_grid=param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
)
grid_bsmote.fit(X_train, y_train)

print('Best params:', grid_bsmote.best_params_)
print('Best CV F1 :', grid_bsmote.best_score_)

best_bsmote = grid_bsmote.best_estimator_

_ = full_evaluation(best_bsmote, X_test, y_test, 'Borderline-SMOTE (training fold only) -- full real test set')


## 6c. ADASYN arm -- same rule: fit only on training fold

`ADASYN` (Adaptive Synthetic Sampling) generates more synthetic points for minority samples that are harder to learn (surrounded by more majority neighbors), rather than a uniform density like SMOTE. It can occasionally fail if a minority sample has zero minority neighbors within its k-neighborhood after feature selection at high tau -- the try/except below reports that clearly instead of crashing the whole loop.

In [ ]:
adasyn_pipeline = ImbPipeline([
    ('prep', preprocessor),
    ('adasyn', ADASYN(random_state=42)),
    ('clf', DecisionTreeClassifier(random_state=42)),
])

grid_adasyn = GridSearchCV(
    estimator=adasyn_pipeline,
    param_grid=param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
)
grid_adasyn.fit(X_train, y_train)

print('Best params:', grid_adasyn.best_params_)
print('Best CV F1 :', grid_adasyn.best_score_)

best_adasyn = grid_adasyn.best_estimator_

_ = full_evaluation(best_adasyn, X_test, y_test, 'ADASYN (training fold only) -- full real test set')


## 7. Transform to named numeric feature space (for SHAP + thresholding)

Uses the baseline model's `prep` step, fit on the training fold only. Real, unresampled data throughout -- resampling never touches what we explain or evaluate on.

In [ ]:
def get_transformed_frame(fitted_prep, X):
    arr = fitted_prep.transform(X)
    names = fitted_prep.get_feature_names_out()
    # strip the ColumnTransformer's 'stepname__' prefix for readability
    clean_names = [n.split('__')[-1] for n in names]
    return pd.DataFrame(arr, columns=clean_names, index=X.index)

fitted_prep_baseline = best_baseline.named_steps['prep']
X_train_t = get_transformed_frame(fitted_prep_baseline, X_train)
X_test_t  = get_transformed_frame(fitted_prep_baseline, X_test)

transformed_feature_names = list(X_train_t.columns)
print(transformed_feature_names)
X_train_t.head()


## 8. SHAP -- global attributions (baseline model)

In [ ]:
import shap

best_model_baseline = best_baseline.named_steps['clf']
explainer = shap.TreeExplainer(best_model_baseline)
shap_values = explainer(X_test_t)
shap_values_positive = shap_values[..., 1]

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_positive, X_test_t,
                   feature_names=transformed_feature_names, show=False)
plt.title('SHAP Beeswarm -- Positive Class (Malaria), gold-standard pipeline')
plt.tight_layout()
plt.savefig('shap_beeswarm_gold_standard.png', dpi=400, bbox_inches='tight')
plt.show()

shap_vals_all = shap_values.values
mean_abs_shap_class_0 = np.mean(np.abs(shap_vals_all[:, :, 0]), axis=0)
mean_abs_shap_class_1 = np.mean(np.abs(shap_vals_all[:, :, 1]), axis=0)
mean_abs_shap_global = (mean_abs_shap_class_0 + mean_abs_shap_class_1) / 2

shap_importance_df = pd.DataFrame({
    'Feature': transformed_feature_names,
    'Mean_Abs_SHAP': mean_abs_shap_global
}).sort_values('Mean_Abs_SHAP', ascending=False)

shap_importance_df


## 9. Tau-thresholding loop -- baseline (no resampling), FULL test set

In [ ]:
def run_threshold_loop(X_train_t, X_test_t, y_train, y_test, mean_abs_shap_global,
                        feature_names, resampler=None, tag=''):
    """resampler: None for no resampling, or an unfit imblearn resampler instance
    (e.g. SMOTE(random_state=42), BorderlineSMOTE(random_state=42)). A fresh clone is
    used inside every threshold's inner CV so nothing is reused across thresholds."""
    from sklearn.base import clone
    thresholds = sorted(np.unique(mean_abs_shap_global))
    results = []

    for t in thresholds:
        mask = mean_abs_shap_global >= float(t)
        selected = [f for f, m in zip(feature_names, mask) if m]
        if len(selected) == 0:
            continue

        X_tr_sel = X_train_t[selected]
        X_te_sel = X_test_t[selected]

        if resampler is not None:
            inner = ImbPipeline([
                ('resample', clone(resampler)),
                ('clf', DecisionTreeClassifier(random_state=42)),
            ])
        else:
            inner = SkPipeline([
                ('clf', DecisionTreeClassifier(random_state=42)),
            ])

        skf_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        try:
            grid_inner = GridSearchCV(
                estimator=inner,
                param_grid={
                    'clf__max_depth': [None, 5, 10, 15, 20],
                    'clf__min_samples_split': [2, 5, 10],
                    'clf__min_samples_leaf': [1, 2, 4],
                    'clf__criterion': ['gini', 'entropy'],
                },
                cv=skf_inner, scoring='f1', n_jobs=-1,
            )
            grid_inner.fit(X_tr_sel, y_train)
        except ValueError as e:
            # ADASYN in particular can fail at high tau (few features left) if a
            # minority sample ends up with zero minority neighbors in that reduced
            # feature space. Report and skip this threshold rather than crashing.
            print(f"[{tag}] tau={t:.4f} | feats={len(selected)} | SKIPPED -- {e}")
            continue
        model_t = grid_inner.best_estimator_

        y_pred_t = model_t.predict(X_te_sel)
        y_proba_t = model_t.predict_proba(X_te_sel)[:, 1]

        report = classification_report(y_test, y_pred_t, output_dict=True)
        results.append({
            'Threshold': float(t),
            'Num_Features': len(selected),
            'Selected_Features': selected,
            'Accuracy': round(accuracy_score(y_test, y_pred_t) * 100, 2),
            'Macro_F1': round(f1_score(y_test, y_pred_t, average='macro'), 4),
            'Dengue_Recall': round(report['0']['recall'], 4),
            'Dengue_Precision': round(report['0']['precision'], 4),
            'ROC_AUC': round(roc_auc_score(y_test, y_proba_t), 4),
            'PR_AUC': round(average_precision_score(y_test, y_proba_t), 4),
        })
        print(f"[{tag}] tau={t:.4f} | feats={len(selected)} | "
              f"MacroF1={results[-1]['Macro_F1']:.4f} | "
              f"DengueRecall={results[-1]['Dengue_Recall']:.4f}")

    return pd.DataFrame(results)

results_baseline = run_threshold_loop(
    X_train_t, X_test_t, y_train, y_test, mean_abs_shap_global,
    transformed_feature_names, resampler=None, tag='baseline'
)
results_baseline


## 10a. Tau-thresholding loop -- WITH SMOTE, FULL real test set

In [ ]:
results_smote = run_threshold_loop(
    X_train_t, X_test_t, y_train, y_test, mean_abs_shap_global,
    transformed_feature_names, resampler=SMOTE(random_state=42), tag='SMOTE'
)
results_smote


## 10b. Tau-thresholding loop -- WITH Borderline-SMOTE, FULL real test set

In [ ]:
results_bsmote = run_threshold_loop(
    X_train_t, X_test_t, y_train, y_test, mean_abs_shap_global,
    transformed_feature_names, resampler=BorderlineSMOTE(random_state=42), tag='BSMOTE'
)
results_bsmote


## 10c. Tau-thresholding loop -- WITH ADASYN, FULL real test set

In [ ]:
results_adasyn = run_threshold_loop(
    X_train_t, X_test_t, y_train, y_test, mean_abs_shap_global,
    transformed_feature_names, resampler=ADASYN(random_state=42), tag='ADASYN'
)
results_adasyn


## 11. RQ1/RQ2/RQ3 comparison table

This is the full design: {all features, XAI-selected} x {no resampling, SMOTE, Borderline-SMOTE, ADASYN}, all computed correctly on the raw data with leakage-safe encoding and full-test-set evaluation.

Note: if ADASYN was skipped at some high-tau thresholds (see warnings above), those rows will be missing from `results_adasyn` -- the merge below uses an outer join so you can still see baseline/SMOTE/BSMOTE results at those thresholds even where ADASYN couldn't run.

In [ ]:
comparison = results_baseline.merge(
    results_smote, on='Threshold', suffixes=('_baseline', '_SMOTE'), how='outer'
).merge(
    results_bsmote.add_suffix('_BSMOTE').rename(columns={'Threshold_BSMOTE': 'Threshold'}),
    on='Threshold', how='outer'
).merge(
    results_adasyn.add_suffix('_ADASYN').rename(columns={'Threshold_ADASYN': 'Threshold'}),
    on='Threshold', how='outer'
).sort_values('Threshold')

comparison.to_csv('gold_standard_comparison_results.csv', index=False)
comparison


## 12. 5-fold CV robustness check (optional, recommended for the paper)

In [ ]:
cv_baseline = SkPipeline([
    ('prep', preprocessor),
    ('clf', DecisionTreeClassifier(
        **{k.replace('clf__', ''): v for k, v in grid_baseline.best_params_.items()},
        random_state=42)),
])

cv_smote = ImbPipeline([
    ('prep', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('clf', DecisionTreeClassifier(
        **{k.replace('clf__', ''): v for k, v in grid_smote.best_params_.items()},
        random_state=42)),
])

cv_bsmote = ImbPipeline([
    ('prep', preprocessor),
    ('bsmote', BorderlineSMOTE(random_state=42)),
    ('clf', DecisionTreeClassifier(
        **{k.replace('clf__', ''): v for k, v in grid_bsmote.best_params_.items()},
        random_state=42)),
])

cv_adasyn = ImbPipeline([
    ('prep', preprocessor),
    ('adasyn', ADASYN(random_state=42)),
    ('clf', DecisionTreeClassifier(
        **{k.replace('clf__', ''): v for k, v in grid_adasyn.best_params_.items()},
        random_state=42)),
])

scoring = {'accuracy': 'accuracy', 'f1_macro': 'f1_macro',
           'roc_auc': 'roc_auc', 'average_precision': 'average_precision'}
cv_split = StratifiedKFold(5, shuffle=True, random_state=42)

for name, pipe in [('Baseline', cv_baseline), ('SMOTE', cv_smote),
                    ('Borderline-SMOTE', cv_bsmote), ('ADASYN', cv_adasyn)]:
    try:
        scores = cross_validate(pipe, X, y, cv=cv_split, scoring=scoring, n_jobs=-1)
        print(f'--- {name} (5-fold CV) ---')
        for metric in scoring:
            vals = scores[f'test_{metric}']
            print(f'  {metric:20s}: {vals.mean():.4f} +/- {vals.std():.4f}')
        print()
    except ValueError as e:
        print(f'--- {name}: FAILED -- {e} ---\n')
